In [9]:
"""
build_shrug_gp_covariates.py
============================
Aggregates SHRUG 2001 village-level Census data to GP level, then merges
to NFHS-4 cluster treatment probabilities for use as pre-treatment controls
in the Rajasthan female political reservation analysis.

GP-level covariates constructed (PAP Section 4.5):
  1. log_pop          - log(total GP population), sum of village populations
  2. scst_share       - SC+ST population share = (SC+ST pop) / total pop
  3. female_pop_share - female population share = female pop / total pop
  4. infra_index      - standardised mean of 5 infrastructure indicators:
                        paved road, school (any), health facility (any),
                        water access (tap/tubewell/handpump), electricity

Linkage chain:
  shrid2 (SHRUG village)
    -> LGD_code / gp_lgd_code  (shrug_LGD_matched.csv)
    -> DHSCLUST                 (cluster_gp_res_long.csv, fully-linked clusters)

Output:
  outputs/final_rj_sample/shrug_gp_covariates.csv
"""

import pandas as pd
import numpy as np
import os

# =============================================================================
# PATHS
# =============================================================================

SHRUG_VD   = ("/Users/sonalideliwala/Library/Mobile Documents/"
              "com~apple~CloudDocs/Desktop/India Data/"
              "shrug 2001 population census village data/"
              "pc01_vd_clean_shrid.dta")

SHRUG_LGD  = ("/Users/sonalideliwala/Documents/GitHub/exemplar/data/"
              "shrug_LGD_matched.csv")

CLUSTER_GP = ("/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/"
              "final_rj_sample/cluster_gp_res_long.csv")

TREAT_FILE = ("/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/"
              "final_rj_sample/cluster_treatment_probs_rj.csv")

OUT_DIR    = "/Users/sonalideliwala/Documents/GitHub/exemplar/outputs/final_rj_sample"
OUT_FILE   = os.path.join(OUT_DIR, "shrug_gp_covariates.csv")

# =============================================================================
# STEP 1: Load SHRUG village directory
# =============================================================================

print("Loading SHRUG village directory...")
vd = pd.read_stata(SHRUG_VD, columns=[
    "shrid2",
    "pc01_vd_t_p",      # total population
    "pc01_vd_t_f",      # female population
    "pc01_vd_sc_p",     # SC population
    "pc01_vd_st_p",     # ST population
    "pc01_vd_app_pr",   # paved road approach
    "pc01_vd_p_sch",    # primary schools
    "pc01_vd_m_sch",    # middle schools
    "pc01_vd_hosp",     # hospitals
    "pc01_vd_ph_cntr",  # primary health centres
    "pc01_vd_tap",      # tap water
    "pc01_vd_tubewell", # tubewell
    "pc01_vd_handpump", # handpump
    "pc01_vd_power_dom",# electricity for domestic use
])
vd["shrid2"] = vd["shrid2"].str.strip()
print(f"  SHRUG villages loaded: {len(vd):,}")

# =============================================================================
# STEP 2: Load shrid2 -> GP LGD crosswalk (Rajasthan only)
# =============================================================================

print("Loading shrid2 -> GP LGD crosswalk...")
xw = pd.read_csv(SHRUG_LGD, dtype=str, encoding='latin-1')
print(f"  Crosswalk rows: {len(xw):,}")

# Keep only Rajasthan Gram Panchayats
xw_rj = xw[xw["state_name"].str.lower().str.strip() == "rajasthan"].copy()
xw_rj = xw_rj[xw_rj["local_body_type"].str.strip().str.lower() == "gram panchayat"]
xw_rj["shrid2"]      = xw_rj["shrid2"].str.strip()
xw_rj["gp_lgd_code"] = xw_rj["LGD_code"].str.strip()
xw_rj = xw_rj.dropna(subset=["gp_lgd_code"])
xw_rj = xw_rj[xw_rj["gp_lgd_code"].isin(["", "nan"]) == False]
print(f"  Rajasthan villages with valid GP LGD code: {len(xw_rj):,}")

# =============================================================================
# STEP 3: Merge SHRUG village data to GP crosswalk
# =============================================================================

print("Merging SHRUG village data to GP crosswalk...")
merged = vd.merge(xw_rj[["shrid2", "gp_lgd_code"]], on="shrid2", how="inner")
print(f"  SHRUG villages matched to GP: {len(merged):,}")
print(f"  Unique GPs covered: {merged['gp_lgd_code'].nunique():,}")

# =============================================================================
# STEP 4: Construct infrastructure components at village level (binary)
# =============================================================================

merged["pop"]    = merged["pc01_vd_t_p"].fillna(0).clip(lower=0)
merged["pop_f"]  = merged["pc01_vd_t_f"].fillna(0).clip(lower=0)
merged["pop_sc"] = merged["pc01_vd_sc_p"].fillna(0).clip(lower=0)
merged["pop_st"] = merged["pc01_vd_st_p"].fillna(0).clip(lower=0)

# Binary infrastructure indicators
merged["has_paved_road"] = (merged["pc01_vd_app_pr"].fillna(0) >= 1).astype(float)
merged["has_school"]     = ((merged["pc01_vd_p_sch"].fillna(0) +
                             merged["pc01_vd_m_sch"].fillna(0)) >= 1).astype(float)
merged["has_health"]     = ((merged["pc01_vd_hosp"].fillna(0) +
                             merged["pc01_vd_ph_cntr"].fillna(0)) >= 1).astype(float)
merged["has_water"]      = ((merged["pc01_vd_tap"].fillna(0) +
                             merged["pc01_vd_tubewell"].fillna(0) +
                             merged["pc01_vd_handpump"].fillna(0)) >= 1).astype(float)
merged["has_elec"]       = (merged["pc01_vd_power_dom"].fillna(0) >= 1).astype(float)

# Population-weighted numerators for infrastructure (rates across villages)
for col in ["has_paved_road", "has_school", "has_health", "has_water", "has_elec"]:
    merged[f"w_{col}"] = merged[col] * merged["pop"]

# =============================================================================
# STEP 5: Aggregate village -> GP level
# Rates: sum of (pop-weighted numerator) / sum of population
# =============================================================================

print("Aggregating village -> GP level...")

gp = merged.groupby("gp_lgd_code").agg(
    gp_total_pop   = ("pop",              "sum"),
    gp_n_villages  = ("shrid2",           "count"),
    sum_pop_f      = ("pop_f",            "sum"),
    sum_pop_sc     = ("pop_sc",           "sum"),
    sum_pop_st     = ("pop_st",           "sum"),
    w_paved_road   = ("w_has_paved_road", "sum"),
    w_school       = ("w_has_school",     "sum"),
    w_health       = ("w_has_health",     "sum"),
    w_water        = ("w_has_water",      "sum"),
    w_elec         = ("w_has_elec",       "sum"),
).reset_index()

# Compute rates (avoid divide-by-zero)
pop = gp["gp_total_pop"].replace(0, np.nan)

gp["female_pop_share"] = gp["sum_pop_f"]  / pop
gp["scst_share"]       = (gp["sum_pop_sc"] + gp["sum_pop_st"]) / pop
gp["share_paved_road"] = gp["w_paved_road"] / pop
gp["share_school"]     = gp["w_school"]     / pop
gp["share_health"]     = gp["w_health"]     / pop
gp["share_water"]      = gp["w_water"]      / pop
gp["share_elec"]       = gp["w_elec"]       / pop

# Log population
gp["log_pop"] = np.log(gp["gp_total_pop"].replace(0, np.nan))

# Infrastructure index: standardise each component on full Rajasthan GP sample, then average
infra_cols = ["share_paved_road", "share_school", "share_health",
              "share_water", "share_elec"]
for col in infra_cols:
    mu = gp[col].mean()
    sd = gp[col].std()
    gp[f"z_{col}"] = (gp[col] - mu) / sd if sd > 0 else 0.0

z_cols = [f"z_{c}" for c in infra_cols]
gp["infra_index"] = gp[z_cols].mean(axis=1)

print(f"  GPs with covariates constructed: {len(gp):,}")

# =============================================================================
# STEP 6: Load cluster -> GP mapping and identify fully-linked clusters
# =============================================================================

print("Loading cluster -> GP mapping...")
cgr = pd.read_csv(CLUSTER_GP)
print(f"  cluster_gp_res_long rows: {len(cgr):,}")

# Identify fully-linked clusters: p_known_treatment_mc == 1
treat = pd.read_csv(TREAT_FILE)
treat.rename(columns={"dhsclust": "DHSCLUST"}, inplace=True)

print(f"\n  Treatment file columns: {list(treat.columns)}")
print(f"  p_known_treatment_mc distribution:")
print(treat["p_known_treatment_mc"].describe())
print(f"  Clusters with p_known_treatment_mc == 1: {(treat['p_known_treatment_mc'] == 1).sum()}")
print(f"  Clusters with p_known_treatment_mc >= 0.99: {(treat['p_known_treatment_mc'] >= 0.99).sum()}")
print(f"  Clusters with p_known_treatment_mc >= 0.95: {(treat['p_known_treatment_mc'] >= 0.95).sum()}")

# Use == 1 as the fully-linked definition per PAP
fully_linked = treat[treat["p_known_treatment_mc"] == 1]["DHSCLUST"].astype(str).tolist()
print(f"\n  Fully-linked clusters (p_known==1): {len(fully_linked)}")

# Filter cluster-GP mapping to fully-linked clusters
cgr["DHSCLUST"]    = cgr["DHSCLUST"].astype(str)
cgr["gp_lgd_code"] = cgr["gp_lgd_code"].astype(str).str.strip()
cgr_linked = cgr[cgr["DHSCLUST"].isin(fully_linked)].copy()
print(f"  cluster-GP rows for fully-linked clusters: {len(cgr_linked):,}")

# =============================================================================
# STEP 7: Merge GP covariates to cluster-GP mapping
# Compute probability-weighted average of GP covariates per cluster
# =============================================================================

print("Merging GP covariates to cluster-GP mapping...")
gp["gp_lgd_code"] = gp["gp_lgd_code"].astype(str).str.strip()

cluster_gp = cgr_linked.merge(
    gp[["gp_lgd_code", "log_pop", "scst_share", "female_pop_share", "infra_index",
        "gp_total_pop", "gp_n_villages"]],
    on="gp_lgd_code",
    how="left"
)

n_matched = cluster_gp["log_pop"].notna().sum()
n_total   = len(cluster_gp)
print(f"  cluster-GP rows with SHRUG covariates: {n_matched:,} / {n_total:,}")

# Probability-weighted average across candidate GPs per cluster
cluster_gp["gp_prob"] = pd.to_numeric(cluster_gp["gp_prob"], errors="coerce")

cov_cols = ["log_pop", "scst_share", "female_pop_share", "infra_index"]
for col in cov_cols:
    cluster_gp[f"w_{col}"] = cluster_gp[col] * cluster_gp["gp_prob"]

cluster_level = cluster_gp.groupby("DHSCLUST").agg(
    **{col: (f"w_{col}", "sum") for col in cov_cols},
    sum_prob      = ("gp_prob",   "sum"),
    n_gps_matched = ("log_pop",   lambda x: x.notna().sum()),
).reset_index()

# Normalise by sum of probabilities
for col in cov_cols:
    cluster_level[col] = cluster_level[col] / cluster_level["sum_prob"]

cluster_level.drop(columns=["sum_prob"], inplace=True)

# =============================================================================
# STEP 8: Diagnostics
# =============================================================================

print(f"\n  Clusters with SHRUG covariates: {len(cluster_level):,}")
print(f"  Clusters with all 4 covariates non-missing: "
      f"{cluster_level[cov_cols].notna().all(axis=1).sum()}")

print("\n--- Covariate summary (cluster level) ---")
print(cluster_level[cov_cols].describe().round(4))

print("\n--- Missing covariate counts ---")
for col in cov_cols:
    n_miss = cluster_level[col].isna().sum()
    print(f"  {col}: {n_miss} missing")

# Coverage check against all fully-linked clusters
n_covered = cluster_level["DHSCLUST"].isin(fully_linked).sum()
print(f"\n  Fully-linked clusters covered: {n_covered} / {len(fully_linked)}")

# Flag any missing
missing_clusters = [c for c in fully_linked if c not in cluster_level["DHSCLUST"].tolist()]
if missing_clusters:
    print(f"  WARNING: {len(missing_clusters)} clusters not covered: {missing_clusters[:10]}")

# =============================================================================
# STEP 9: Save
# =============================================================================

out = cluster_level.copy()
out["DHSCLUST"] = out["DHSCLUST"].astype(int)
out = out.sort_values("DHSCLUST").reset_index(drop=True)

out.to_csv(OUT_FILE, index=False)
print(f"\nSaved: {OUT_FILE}")
print(f"Columns: {list(out.columns)}")
print(out.head(10).to_string())


Loading SHRUG village directory...
  SHRUG villages loaded: 527,920
Loading shrid2 -> GP LGD crosswalk...
  Crosswalk rows: 596,389
  Rajasthan villages with valid GP LGD code: 39,411
Merging SHRUG village data to GP crosswalk...
  SHRUG villages matched to GP: 39,048
  Unique GPs covered: 11,085
Aggregating village -> GP level...
  GPs with covariates constructed: 11,085
Loading cluster -> GP mapping...
  cluster_gp_res_long rows: 9,733

  Treatment file columns: ['DHSCLUST', 'DHSREGNA', 'p_never_mc', 'p_once_mc', 'p_twice_mc', 'p_any_reserved_mc', 'expected_dose_mc', 'p_known_treatment_mc', 'p_unknown_treatment_mc', 'treatment_certainty_mc', 'primary_gp', 'primary_gp_prob', 'primary_gp_dose', 'n_gps_hit']
  p_known_treatment_mc distribution:
count    732.000000
mean       0.808951
std        0.164147
min        0.322906
25%        0.689713
50%        0.835790
75%        0.961189
max        1.000000
Name: p_known_treatment_mc, dtype: float64
  Clusters with p_known_treatment_mc == 1: 

In [7]:
import pandas as pd
xw2 = pd.read_csv("/Users/sonalideliwala/Documents/GitHub/exemplar/data/shrug_LGD_matched.csv", dtype=str, encoding='latin-1', nrows=5)
print(xw2.columns.tolist())
print(xw2.head(5).to_string())

['pc11_shrid_id', 'shrid2', 'state_name', 'district_name', 'subdistrict_name', 'town_name', 'village_name', 'place_name', 'LGD_code', 'local_body_name', 'local_body_type', 'gp_to_urban_conversion', 'old_gp_lgd_code', 'old_gp_name', 'shrid_part_of_multi_village_GP']
  pc11_shrid_id                  shrid2     state_name district_name subdistrict_name town_name village_name place_name LGD_code                                               local_body_name                                               local_body_type gp_to_urban_conversion old_gp_lgd_code old_gp_name shrid_part_of_multi_village_GP
0        000001  11-01-001-00001-000001  jammu kashmir       kupwara          kupwara       NaN         bore       bore     7237                                                       Pathran                                                Gram Panchayat                     No             NaN         NaN                            Yes
1        000002  11-01-001-00001-000002  jammu kashmir       kup